<a href="https://colab.research.google.com/github/niharikakt024/AI-Agent-for-Data-Cleaning/blob/main/universal_data_cleaning_agent_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧹 Universal AI Data-Cleaning Agent

Upload **any** messy CSV and this agent will:
1. Profile every column (type, missing %, outliers, samples)
2. Ask an LLM (Groq / Llama 3.3 70B) to build a cleaning plan
3. Automatically apply fixes: whitespace, casing, fuzzy category merging, date parsing, currency cleaning, invalid negatives, outlier capping, boolean normalization, missing-value imputation
4. Detect and remove duplicates
5. Generate a before/after report and download the cleaned file

**Setup needed once:** a free Groq API key saved as a Colab secret named `GROQ_API_KEY`.
Get one at console.groq.com → API Keys → Create API Key.

In [1]:
# Cell 1 — Install dependencies
!pip install groq pandas numpy thefuzz python-Levenshtein -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 40.2 MB/s eta 0:00:00


In [2]:
# Cell 2 — Connect to Groq
from google.colab import userdata
from groq import Groq
import pandas as pd
import numpy as np
import json
import re
import calendar
from thefuzz import fuzz

client = Groq(api_key=userdata.get('GROQ_API_KEY'))

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say hello in 5 words"}]
)
print("Connection test:", response.choices[0].message.content)

Connection test: Hello, how are you today?


In [3]:
# Cell 3 — Upload any CSV
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print(f"Shape: {df.shape}")
df.head()

Saving dirty_financial_transactions.csv to dirty_financial_transactions.csv
Shape: (100000, 8)


,Transaction_ID,Transaction_Date,Customer_ID,Product_Name,Quantity,Price,Payment_Method,Transaction_Status
0,T0001,2024-08-02,C2205,Headphones,-5.0,$420.21,pay pal,NaN
1,T0002,2020-02-10,C3156,Coffee,469.0,-445.34202525395585,creditcard,Pending
2,T0003,2025-02-30,C2919,Tablet,-4.0,810.9930123946459,credit card,completed
3,T0004,2020-08-17,C3009,Tab,-7.0,868.6083413217348,PayPal,Pending
4,T0005,2025-02-30,C3488,Coffee Machine,-10.0,-763.1224490039416,PayPal,completed


## Step 1 — Universal Profiler

Works on any dataset. Detects missing values, data type, uniqueness, numeric outliers, invalid negatives, currency-like text, extra whitespace, and boolean-like columns automatically.

In [17]:
# Cell 4 — Universal profiler
def profile_dataframe(df):
    profile = {}
    for col in df.columns:
        col_data = df[col]
        non_null = col_data.dropna()

        info = {
            "dtype": str(col_data.dtype),
            "missing_pct": round(col_data.isna().mean() * 100, 2),
            "unique_count": int(col_data.nunique()),
            "sample_values": non_null.astype(str).unique()[:8].tolist()
        }

        # Detect boolean-like columns (Yes/No, True/False, 1/0, Y/N)
        vals_lower = set(str(v).strip().lower() for v in non_null.unique()[:20])
        bool_like_sets = [
            {"yes", "no"}, {"true", "false"}, {"y", "n"}, {"1", "0"}, {"1.0", "0.0"}
        ]
        info["boolean_like"] = any(vals_lower.issubset(s) for s in bool_like_sets) and len(vals_lower) <= 2

        # Numeric-specific checks
        numeric_col = pd.to_numeric(col_data, errors="coerce")
        pct_numeric_parseable = numeric_col.notna().mean()
        if pd.api.types.is_numeric_dtype(col_data) or pct_numeric_parseable > 0.8:
            valid_numeric = numeric_col.dropna()
            if len(valid_numeric) > 0:
                q1, q3 = valid_numeric.quantile(0.25), valid_numeric.quantile(0.75)
                iqr = q3 - q1
                lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
                outlier_count = int(((valid_numeric < lower_bound) | (valid_numeric > upper_bound)).sum())
                info["min"] = float(valid_numeric.min())
                info["max"] = float(valid_numeric.max())
                info["has_negative"] = bool((valid_numeric < 0).any())
                info["outlier_count_iqr"] = outlier_count
                info["stored_as_text_but_numeric"] = not pd.api.types.is_numeric_dtype(col_data)

        # Detect currency-like text (contains $ or thousand-separator commas)
        if col_data.dtype == object:
            sample_str = " ".join(non_null.astype(str).head(20))
            info["looks_like_currency"] = bool(re.search(r"[$]|\d,\d{3}", sample_str))
            info["has_extra_whitespace"] = bool(non_null.astype(str).str.strip().ne(non_null.astype(str)).any())

        profile[col] = info
    return profile

profile = profile_dataframe(df)
print(json.dumps(profile, indent=2, default=str))

{
  "Transaction_ID": {
    "dtype": "object",
    "missing_pct": 5.02,
    "unique_count": 94040,
    "sample_values": [
      "T0001",
      "T0002",
      "T0003",
      "T0004",
      "T0005",
      "T0006",
      "T0008",
      "T0009"
    ],
    "boolean_like": false,
    "looks_like_currency": false,
    "has_extra_whitespace": false
  },
  "Transaction_Date": {
    "dtype": "object",
    "missing_pct": 4.88,
    "unique_count": 1861,
    "sample_values": [
      "2024-08-02",
      "2020-02-10",
      "2025-02-30",
      "2020-08-17",
      "2021-10-26",
      "2023-13-01",
      "2020-03-18",
      "2020-06-19"
    ],
    "boolean_like": false,
    "looks_like_currency": false,
    "has_extra_whitespace": false
  },
  "Customer_ID": {
    "dtype": "object",
    "missing_pct": 4.88,
    "unique_count": 5000,
    "sample_values": [
      "C2205",
      "C3156",
      "C2919",
      "C3009",
      "C3488",
      "C4241",
      "C1313",
      "C4736"
    ],
    "boolean_like": fal

## Step 2 — LLM Cleaning Plan

The agent reasons over the profile and returns a structured plan of actions per column, with a wide action vocabulary covering most common real-world messiness.

In [18]:
# Cell 5 — Ask the LLM for a cleaning plan
ACTION_LIST = """
- trim_whitespace: strip leading/trailing spaces from text
- standardize_case: fix inconsistent capitalization (Title Case)
- fuzzy_merge_categories: merge near-duplicate spellings of the same category (e.g. "Paypal" vs "Pay Pal")
- parse_dates: convert inconsistent/invalid date strings into a standard datetime format, repairing swapped month/day and out-of-range days instead of discarding them
- clean_currency: strip $ and commas from a numeric-looking text column, convert to numeric
- fix_invalid_negative: convert negative values to absolute value (for columns like quantity/price/age that can't logically be negative but the sign looks like a data-entry error)
- clip_negative_to_zero: clamp negative values to 0 (for columns like discounts/refunds/inventory-adjustments where a negative reading means "none" rather than a sign error)
- cap_outliers_iqr: cap extreme outlier values to the IQR bounds instead of deleting them
- cap_outliers_zscore: cap extreme outlier values beyond 3 standard deviations from the mean
- standardize_boolean: convert Yes/No, Y/N, True/False, 1/0 variants into a consistent True/False
- fill_missing_median: fill missing numeric values with the column median
- fill_missing_mode: fill missing categorical values with the column mode
- fill_missing_constant: fill missing values with a sensible constant (0 for numeric columns, "Unknown" for text columns) when median/mode imputation doesn't make sense
- remove_special_characters: strip punctuation/symbols from a text column, keeping only letters, numbers, and spaces
- remove_non_ascii: strip emojis and non-ASCII characters from a text column
- standardize_id_format: uppercase an ID/code column and remove internal spaces so IDs match consistently
- standardize_email: lowercase and trim email addresses, and null out values that aren't validly formatted
- standardize_phone: strip all non-digit characters from a phone number column and keep the last 10 digits
- standardize_url: lowercase a URL column and strip trailing slashes
- extract_numeric: pull the numeric value out of a column that mixes numbers with text/units (e.g. "5 units", "3kg")
- round_numeric: round a numeric column (e.g. price, percentage) to 2 decimal places
- drop_column: drop a column entirely (e.g. almost entirely missing, or clearly not useful for analysis)
- remove_negative_sign: strip the negative sign from a price/amount column, converting negative values to their absolute value
- none: no action needed
"""

def get_cleaning_plan(profile):
    prompt = f"""You are an expert data cleaning agent. Given this column profile from a pandas DataFrame,
decide the best cleaning action(s) for EACH column.

Available actions:
{ACTION_LIST}

PROFILE:
{json.dumps(profile, indent=2, default=str)}

Rules:
- Only suggest fix_invalid_negative if has_negative is true AND the column is something that logically cannot be negative (e.g. price, quantity, age, salary) - judge this from the column name.
- Only suggest clip_negative_to_zero instead of fix_invalid_negative when a negative value more plausibly means "zero/none" than a flipped sign (e.g. discount, refund, adjustment columns) - pick one or the other, never both.
- Only suggest cap_outliers_iqr or cap_outliers_zscore if outlier_count_iqr is meaningfully large relative to dataset size - pick whichever is more appropriate, not both.
- Only suggest clean_currency if looks_like_currency is true.
- Only suggest standardize_boolean if boolean_like is true.
- Only suggest standardize_email / standardize_phone / standardize_url if the column name or sample values clearly indicate that content type.
- Only suggest extract_numeric if the column mixes numbers with units or other text that clean_currency wouldn't handle.
- Only suggest drop_column if missing_pct is extremely high (e.g. above 90%) or the column is clearly an unusable artifact - use sparingly, this is destructive.
- A column can have multiple actions in the list, in the order they should be applied.

Return ONLY valid JSON (no markdown, no extra commentary) in this exact structure:
{{
  "column_name": {{
    "issue": "short description",
    "actions": ["action1", "action2"],
    "reasoning": "one sentence why"
  }}
}}"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    text = response.choices[0].message.content.replace("```json", "").replace("```", "").strip()
    return json.loads(text)

plan = get_cleaning_plan(profile)
print(json.dumps(plan, indent=2))

{
  "Transaction_ID": {
    "issue": "missing values and possible inconsistent ID formatting",
    "actions": [
      "fill_missing_constant",
      "standardize_id_format"
    ],
    "reasoning": "Because the column is an ID and has missing values, it should be filled with a constant and then standardized to ensure consistent formatting."
  },
  "Transaction_Date": {
    "issue": "inconsistent date strings",
    "actions": [
      "parse_dates"
    ],
    "reasoning": "The column contains inconsistent date strings that need to be parsed into a standard datetime format."
  },
  "Customer_ID": {
    "issue": "missing values and possible inconsistent ID formatting",
    "actions": [
      "fill_missing_constant",
      "standardize_id_format"
    ],
    "reasoning": "Because the column is an ID and has missing values, it should be filled with a constant and then standardized to ensure consistent formatting."
  },
  "Product_Name": {
    "issue": "extra whitespace",
    "actions": [
     

## Step 3 — Universal Executor

No hardcoded column names anywhere. Every function below works generically on whatever column the LLM points it at, for any dataset.

In [19]:
# Cell 6 — Universal executor
def fuzzy_merge_categories(series, threshold=85):
    series = series.astype(str).str.strip()
    counts = series.value_counts()
    vals = counts.index.tolist()
    mapping, assigned = {}, set()
    for val in vals:
        if val in assigned:
            continue
        group = [val]
        assigned.add(val)
        for other in vals:
            if other not in assigned and fuzz.token_sort_ratio(val.lower(), other.lower()) >= threshold:
                group.append(other)
                assigned.add(other)
        canonical = counts[group].idxmax()
        for m in group:
            mapping[m] = canonical
    return series.map(mapping)


def standardize_boolean(series):
    mapping = {
        "yes": True, "no": False, "y": True, "n": False,
        "true": True, "false": False, "1": True, "0": False,
        "1.0": True, "0.0": False
    }
    return series.astype(str).str.strip().str.lower().map(mapping)


def cap_outliers_iqr(series):
    numeric = pd.to_numeric(series, errors="coerce")
    q1, q3 = numeric.quantile(0.25), numeric.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return numeric.clip(lower=lower, upper=upper)


def cap_outliers_zscore(series, threshold=3):
    numeric = pd.to_numeric(series, errors="coerce")
    mean, std = numeric.mean(), numeric.std()
    if not std or pd.isna(std):
        return numeric
    lower, upper = mean - threshold * std, mean + threshold * std
    return numeric.clip(lower=lower, upper=upper)


def standardize_email(series):
    s = series.astype(str).str.strip().str.lower()
    valid = s.str.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
    return s.where(valid, np.nan)


def standardize_phone(series):
    digits = series.astype(str).str.replace(r"\D", "", regex=True)
    digits = digits.replace("", np.nan)
    return digits.apply(lambda d: d if pd.isna(d) else d[-10:])


def extract_numeric(series):
    extracted = series.astype(str).str.extract(r"(-?\d+\.?\d*)")[0]
    return pd.to_numeric(extracted, errors="coerce")


def smart_parse_date(date_str):
    """
    Parse a date string, repairing common data-entry mistakes instead of
    discarding the row:
      - month/day swapped (e.g. "2023-13-01" -> month 13 is invalid, so it's
        really day=13, month=01 -> 2023-01-13)
      - day-of-month overflow for the given month (e.g. "2025-02-30" -> Feb
        doesn't have 30 days, clamp to the real last day -> 2025-02-28)
    Only returns NaT if the string truly can't be interpreted as a date
    (e.g. no valid 4-digit year present).
    """
    if pd.isna(date_str):
        return pd.NaT

    s = str(date_str).strip()

    # 1) try pandas' own flexible parser first — handles anything already valid
    parsed = pd.to_datetime(s, errors="coerce", format="mixed")
    if pd.notna(parsed):
        return parsed

    # 2) fall back to manual repair of the numeric components
    parts = re.findall(r"\d+", s)
    if len(parts) != 3:
        return pd.NaT

    year_idx = next((i for i, p in enumerate(parts) if len(p) == 4), None)
    if year_idx is None:
        return pd.NaT
    year = int(parts[year_idx])
    remaining = [int(p) for i, p in enumerate(parts) if i != year_idx]
    month, day = remaining[0], remaining[1]

    # repair: if month is out of range but day isn't, they were swapped
    if month > 12 and day <= 12:
        month, day = day, month

    if not (1 <= month <= 12):
        return pd.NaT

    # repair: clamp an overflowing day to the last real day of that month
    last_day = calendar.monthrange(year, month)[1]
    day = min(max(day, 1), last_day)

    try:
        return pd.Timestamp(year=year, month=month, day=day)
    except ValueError:
        return pd.NaT


def apply_cleaning_plan(df, plan):
    df_clean = df.copy()
    log = []

    for col, spec in plan.items():
        if col not in df_clean.columns:
            continue
        actions = spec.get("actions", [])
        if isinstance(actions, str):
            actions = [actions]

        for action in actions:
            action = action.lower().strip()

            if action == "trim_whitespace":
                df_clean[col] = df_clean[col].astype(str).str.strip()
                log.append(f"{col}: trimmed extra whitespace")

            elif action == "standardize_case":
                df_clean[col] = df_clean[col].astype(str).str.strip().str.title()
                df_clean[col] = df_clean[col].replace("Nan", np.nan)
                log.append(f"{col}: standardized text casing")

            elif action == "fuzzy_merge_categories":
                df_clean[col] = fuzzy_merge_categories(df_clean[col])
                log.append(f"{col}: merged near-duplicate category spellings")

            elif action == "parse_dates":
                df_clean[col] = df_clean[col].apply(smart_parse_date)
                log.append(f"{col}: parsed into consistent datetime format (repaired swapped month/day and out-of-range days instead of discarding them)")

            elif action == "clean_currency":
                cleaned = df_clean[col].astype(str).str.replace(r"[$,]", "", regex=True)
                df_clean[col] = pd.to_numeric(cleaned, errors="coerce")
                log.append(f"{col}: cleaned currency symbols, converted to numeric")

            elif action == "fix_invalid_negative":
                df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").abs()
                log.append(f"{col}: converted invalid negative values to absolute value")

            elif action == "cap_outliers_iqr":
                df_clean[col] = cap_outliers_iqr(df_clean[col])
                log.append(f"{col}: capped extreme outliers to IQR bounds")

            elif action == "standardize_boolean":
                df_clean[col] = standardize_boolean(df_clean[col])
                log.append(f"{col}: standardized boolean-like values to True/False")

            elif action == "fill_missing_median":
                numeric_col = pd.to_numeric(df_clean[col], errors="coerce")
                med = numeric_col.median()
                df_clean[col] = numeric_col.fillna(med)
                log.append(f"{col}: filled missing values with median ({round(med, 2)})")

            elif action == "fill_missing_mode":
                if df_clean[col].notna().any():
                    mode_val = df_clean[col].mode()[0]
                    df_clean[col] = df_clean[col].fillna(mode_val)
                    log.append(f"{col}: filled missing values with mode ('{mode_val}')")

            elif action == "fill_missing_constant":
                numeric_col = pd.to_numeric(df_clean[col], errors="coerce")
                if numeric_col.notna().sum() >= df_clean[col].notna().sum() * 0.9:
                    df_clean[col] = numeric_col.fillna(0)
                    log.append(f"{col}: filled missing values with constant 0")
                else:
                    df_clean[col] = df_clean[col].fillna("Unknown")
                    log.append(f"{col}: filled missing values with constant 'Unknown'")

            elif action == "clip_negative_to_zero":
                df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").clip(lower=0)
                log.append(f"{col}: clipped negative values to 0")

            elif action == "cap_outliers_zscore":
                df_clean[col] = cap_outliers_zscore(df_clean[col])
                log.append(f"{col}: capped outliers beyond 3 standard deviations")

            elif action == "remove_special_characters":
                df_clean[col] = df_clean[col].astype(str).str.replace(r"[^A-Za-z0-9\s]", "", regex=True).str.strip()
                log.append(f"{col}: removed special characters")

            elif action == "remove_non_ascii":
                df_clean[col] = (
                    df_clean[col].astype(str)
                    .str.encode("ascii", "ignore").str.decode("ascii")
                    .str.strip()
                )
                log.append(f"{col}: removed non-ASCII characters")

            elif action == "standardize_id_format":
                df_clean[col] = df_clean[col].astype(str).str.upper().str.replace(r"\s+", "", regex=True)
                log.append(f"{col}: standardized ID format (uppercase, no spaces)")

            elif action == "standardize_email":
                df_clean[col] = standardize_email(df_clean[col])
                log.append(f"{col}: lowercased emails and nulled malformed addresses")

            elif action == "standardize_phone":
                df_clean[col] = standardize_phone(df_clean[col])
                log.append(f"{col}: stripped formatting, kept last 10 digits of phone numbers")

            elif action == "standardize_url":
                df_clean[col] = df_clean[col].astype(str).str.strip().str.lower().str.rstrip("/")
                log.append(f"{col}: standardized URL casing and trailing slashes")

            elif action == "extract_numeric":
                df_clean[col] = extract_numeric(df_clean[col])
                log.append(f"{col}: extracted numeric value from mixed text")

            elif action == "round_numeric":
                df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").round(2)
                log.append(f"{col}: rounded to 2 decimal places")

            elif action == "drop_column":
                df_clean = df_clean.drop(columns=[col])
                log.append(f"{col}: dropped column (too sparse / not useful)")
                break

            elif action == "remove_negative_sign":
                df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").abs()
                log.append(f"{col}: removed negative sign, converted to absolute value")

    return df_clean, log

df_clean, log = apply_cleaning_plan(df, plan)
print("=== CLEANING LOG ===")
for entry in log:
    print("-", entry)
df_clean.head(10)

=== CLEANING LOG ===
- Transaction_ID: filled missing values with constant 'Unknown'
- Transaction_ID: standardized ID format (uppercase, no spaces)
- Transaction_Date: parsed into consistent datetime format (repaired swapped month/day and out-of-range days instead of discarding them)
- Customer_ID: filled missing values with constant 'Unknown'
- Customer_ID: standardized ID format (uppercase, no spaces)
- Product_Name: trimmed extra whitespace
- Product_Name: standardized text casing
- Quantity: converted invalid negative values to absolute value
- Price: cleaned currency symbols, converted to numeric
- Price: converted invalid negative values to absolute value
- Payment_Method: trimmed extra whitespace
- Payment_Method: merged near-duplicate category spellings
- Payment_Method: standardized text casing
- Transaction_Status: standardized text casing


,Transaction_ID,Transaction_Date,Customer_ID,Product_Name,Quantity,Price,Payment_Method,Transaction_Status
0,T0001,2024-08-02,C2205,Headphones,5.0,420.210000,Pay Pal,NaN
1,T0002,2020-02-10,C3156,Coffee,469.0,445.342025,Creditcard,Pending
2,T0003,2025-02-28,C2919,Tablet,4.0,810.993012,Credit Card,Completed
3,T0004,2020-08-17,C3009,Tab,7.0,868.608341,Paypal,Pending
4,T0005,2025-02-28,C3488,Coffee Machine,10.0,763.122449,Paypal,Completed
5,T0006,2021-10-26,C4241,Smartphone,598.0,NaN,Paypal,Completed
6,UNKNOWN,2025-02-28,C1313,Laptop,10.0,NaN,Credit Card,Completed
7,T0008,2023-01-13,C4736,Headphones,669.0,86.921269,Cash,NaN
8,T0009,NaT,C3387,Tablet,10.0,461.701984,Paypal,NaN
9,T0010,2025-02-28,C2846,Laptop,1.0,404.890707,Creditcard,Pending


## Step 4 — Duplicate Detection

Checks for fully identical rows — works regardless of schema, no hardcoded ID column needed.

In [16]:
# Cell 7 — Remove duplicates
before = len(df_clean)

df_clean = df_clean.drop_duplicates(keep="first")
after = len(df_clean)
print(f"Removed {before - after} fully duplicate rows")
print(f"Final shape: {df_clean.shape}")

Removed 994 fully duplicate rows
Final shape: (99006, 8)


## Step 5 — Report & Download

In [12]:
# Cell 8 — Generate report and download cleaned file
report_lines = []
report_lines.append("DATA CLEANING AGENT - SUMMARY REPORT")
report_lines.append("=" * 50)
report_lines.append(f"Original rows: {len(df)} | Final rows: {len(df_clean)}")
report_lines.append(f"Missing values before: {int(df.isna().sum().sum())} | after: {int(df_clean.isna().sum().sum())}")
report_lines.append("")
report_lines.append("Actions taken:")
for entry in log:
    report_lines.append(f"  - {entry}")
report_lines.append(f"  - {before - after} fully duplicate rows removed")
report_lines.append("=" * 50)

report = "\n".join(report_lines)
print(report)

df_clean.to_csv("cleaned_data.csv", index=False)
with open("cleaning_report.txt", "w") as f:
    f.write(report)

files.download("cleaned_data.csv")
files.download("cleaning_report.txt")

DATA CLEANING AGENT - SUMMARY REPORT
Original rows: 100000 | Final rows: 99006
Missing values before: 69971 | after: 36110

Actions taken:
  - Transaction_Date: parsed into consistent datetime format (repaired swapped month/day and out-of-range days instead of discarding them)
  - Product_Name: trimmed extra whitespace
  - Product_Name: standardized text casing
  - Quantity: converted invalid negative values to absolute value
  - Quantity: capped extreme outliers to IQR bounds
  - Price: cleaned currency symbols, converted to numeric
  - Price: filled missing values with median (51.61)
  - Payment_Method: trimmed extra whitespace
  - Payment_Method: merged near-duplicate category spellings
  - Payment_Method: standardized text casing
  - Transaction_Status: standardized text casing
  - 994 fully duplicate rows removed


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>